In [ ]:
from vosk import Model, KaldiRecognizer
import pyaudio
import threading
import queue
import socket
# Laptop's IP and port for sending commands
LAPTOP_PORT = 12345
ESP32_IP = "192.168.1.106"
result_queue = queue.Queue()

english_model = Model(r"vosk-model-small-en-us-0.15\vosk-model-small-en-us-0.15")  
arabic_model= Model(r"vosk-model-ar-mgb2-0.4\vosk-model-ar-mgb2-0.4")  
recognizer = KaldiRecognizer(arabic_model, 16000)
trigger_actions = {
    ("دام", "أمام","forward",): lambda: result_queue.put("W"),
    
    ("يمين","right",): lambda: result_queue.put("P"),
    ("شمال", "يسار","left",): lambda: result_queue.put("A"),
    ("نص","منتصف","str",): lambda:result_queue.put("H"),
    ("قف","وقف","stop",): lambda:result_queue.put("Q"),
    ("فتح","open",): lambda: result_queue.put("E"),
    ("غلق","قفل","close",):lambda: result_queue.put("R"),
    ("ورا", "خلف", "وره", "ورى","back",): lambda:print("Trigger word detected (ورا/خلف)!"),
}

mic = pyaudio.PyAudio()
stream = mic.open(rate=16000, channels=1, format=pyaudio.paInt16, input=True, frames_per_buffer=8192)

def send_commands():
    while True:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.connect((ESP32_IP, LAPTOP_PORT))
            print(f"Connected to {ESP32_IP}:{LAPTOP_PORT}")
        
            while True:
                
                try:
                    command= result_queue.get()
                    if command is None:
                        break
                    s.sendall(command.encode()+b'\n')
                    print(f"Sent {command} to ESP32-CAM")
                    
                except Exception as e:
                    print(f"Error: {e}")
                    break


def listen_for_commands():
    while True:
        # print("listening...")
        data = stream.read(8000)
        if recognizer.AcceptWaveform(data):
            result = recognizer.Result()
            print(result)
            for words, action in trigger_actions.items():
                if any(word in result for word in words):
                    action()
                    break  # Optional: Stop after first match

thread_listen = threading.Thread(target=listen_for_commands, daemon=True) # control the thread in the background 
thread_listen.start()
thread_send_commands = threading.Thread(target=send_commands, daemon=True)
thread_send_commands.start()
thread_listen.join()
thread_send_commands.join()



Connected to 192.168.1.106:12345
{
  "text" : "تحرك له"
}
{
  "text" : "تحرك لأقدام"
}
Sent W to ESP32-CAM
{
  "text" : "تحرك للخلف"
}
Trigger word detected (ورا/خلف)!
{
  "text" : "تحرك للتوقف"
}
Sent Q to ESP32-CAM
{
  "text" : "رحيمين"
}
Sent P to ESP32-CAM
{
  "text" : "بروح"
}
{
  "text" : "روح شمال"
}
Sent A to ESP32-CAM
{
  "text" : "روح للنص"
}
Sent H to ESP32-CAM
{
  "text" : "افتح الكشيف"
}
Sent E to ESP32-CAM
{
  "text" : "أغلق"
}
Sent R to ESP32-CAM
{
  "text" : ""
}
{
  "text" : "ما"
}
{
  "text" : ""
}
{
  "text" : ""
}


In [ ]:
import speech_recognition as sr

r = sr.Recognizer()
with sr.Microphone() as source:
    while True:
        r.adjust_for_ambient_noise(source)
        audio = r.listen(source)
        print("Listening...")
        try:
            
            text = r.recognize_google(audio).lower()
            print(text)
            # if "hi" in text.lower():
            #     print("Trigger word detected!")
                # Perform action
        except:
            pass

Listening...
Listening...
Listening...
hi
Listening...
Listening...
hi
Listening...
Listening...
how are you
Listening...
what
Listening...
Listening...
what are
Listening...
Listening...
Listening...
Listening...
Listening...
Listening...
Listening...
